### 1. Konfiguracja bazy danych SQLite

Pierwszym krokiem jest stworzenie bazy danych SQLite i tabeli `patients` do przechowywania danych pacjentów. Tabela będzie zawierać pola takie jak `id`, `name`, `surname`, `pesel`, `password`.

In [7]:
import sqlite3
import hashlib
import os

def init_db():
    # Usunięcie istniejącej bazy danych, aby móc utworzyć nowy schemat
    if os.path.exists('patients.db'):
        os.remove('patients.db')
        print("Usunięto istniejącą bazę danych 'patients.db'.")

    conn = sqlite3.connect('patients.db')
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Pacjent (
            pacjent_id INTEGER PRIMARY KEY AUTOINCREMENT,
            imie VARCHAR(50),
            nazwisko VARCHAR(50),
            pesel VARCHAR(11) UNIQUE,
            data_urodzenia DATE,
            telefon VARCHAR(15),
            email VARCHAR(100)
        );
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Lekarz (
            lekarz_id INTEGER PRIMARY KEY AUTOINCREMENT,
            imie VARCHAR(50),
            nazwisko VARCHAR(50),
            specjalizacja VARCHAR(100),
            nr_prawa_wykonywania_zawodu VARCHAR(50)
        );
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Placowka (
            placowka_id INTEGER PRIMARY KEY AUTOINCREMENT,
            nazwa VARCHAR(100),
            adres VARCHAR(200),
            telefon VARCHAR(15)
        );
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Wizyta (
            wizyta_id INTEGER PRIMARY KEY AUTOINCREMENT,
            data DATE,
            godzina TIME,
            status VARCHAR(50),
            pacjent_id INT,
            lekarz_id INT,
            placowka_id INT,
            FOREIGN KEY (pacjent_id) REFERENCES Pacjent(pacjent_id),
            FOREIGN KEY (lekarz_id) REFERENCES Lekarz(lekarz_id),
            FOREIGN KEY (placowka_id) REFERENCES Placowka(placowka_id)
        );
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Dokumentacja_medyczna (
            dokument_id INTEGER PRIMARY KEY AUTOINCREMENT,
            opis TEXT,
            data_utworzenia DATE,
            wizyta_id INT UNIQUE,
            FOREIGN KEY (wizyta_id) REFERENCES Wizyta(wizyta_id)
        );
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Platnosc (
            platnosc_id INTEGER PRIMARY KEY AUTOINCREMENT,
            kwota DECIMAL(10,2),
            status VARCHAR(50),
            metoda VARCHAR(50),
            wizyta_id INT UNIQUE,
            FOREIGN KEY (wizyta_id) REFERENCES Wizyta(wizyta_id)
        );
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Wynik_badania (
            wynik_id INTEGER PRIMARY KEY AUTOINCREMENT,
            typ_badania VARCHAR(100),
            opis TEXT,
            plik VARCHAR(255),
            dokument_id INT,
            FOREIGN KEY (dokument_id) REFERENCES Dokumentacja_medyczna(dokument_id)
        );
    ''')

    conn.commit()
    conn.close()
    print("Baza danych 'patients.db' i nowe tabele zostały zainicjowane.")

# Inicjalizacja bazy danych przy pierwszym uruchomieniu
init_db()

Usunięto istniejącą bazę danych 'patients.db'.
Baza danych 'patients.db' i nowe tabele zostały zainicjowane.


### 2. Interfejs HTML formularza do dodania nowego pacjenta

Oto podstawowy formularz HTML, który pozwoli na wprowadzenie danych nowego pacjenta. Aby przetworzyć ten formularz w rzeczywistej aplikacji webowej, musiałby być on obsłużony przez backend (np. Flask).

```html
<!DOCTYPE html>
<html>
<head>
    <title>Dodaj Nowego Pacjenta</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; }
        .container { background-color: #fff; padding: 20px; border-radius: 8px; box-shadow: 0 0 10px rgba(0,0,0,0.1); max-width: 500px; margin: auto; }
        h2 { color: #333; }
        label { display: block; margin-bottom: 8px; font-weight: bold; }
        input[type="text"], input[type="email"], input[type="date"], input[type="tel"] { width: calc(100% - 22px); padding: 10px; margin-bottom: 15px; border: 1px solid #ddd; border-radius: 4px; }
        input[type="submit"] { background-color: #4CAF50; color: white; padding: 10px 15px; border: none; border-radius: 4px; cursor: pointer; font-size: 16px; }
        input[type="submit"]:hover { background-color: #45a049; }
    </style>
</head>
<body>
    <div class="container">
        <h2>Dodaj Nowego Pacjenta</h2>
        <form action="/add_patient" method="post">
            <label for="imie">Imię:</label>
            <input type="text" id="imie" name="imie" required>

            <label for="nazwisko">Nazwisko:</label>
            <input type="text" id="nazwisko" name="nazwisko" required>

            <label for="pesel">PESEL:</label>
            <input type="text" id="pesel" name="pesel" pattern="[0-9]{11}" title="PESEL musi składać się z 11 cyfr" required>

            <label for="data_urodzenia">Data Urodzenia:</label>
            <input type="date" id="data_urodzenia" name="data_urodzenia">

            <label for="telefon">Telefon:</label>
            <input type="tel" id="telefon" name="telefon" pattern="[0-9]{9}" title="Numer telefonu musi składać się z 9 cyfr">

            <label for="email">Email:</label>
            <input type="email" id="email" name="email">

            <input type="submit" value="Dodaj Pacjenta">
        </form>
    </div>
</body>
</html>
```

### 2.1 Zapisywanie danych z formularza do bazy wbudowanej

Poniższa funkcja Pythona symuluje zapis danych z formularza do bazy danych. W rzeczywistej aplikacji byłaby wywoływana po przesłaniu formularza HTML.

In [8]:
def add_patient(imie, nazwisko, pesel, data_urodzenia=None, telefon=None, email=None):
    conn = sqlite3.connect('patients.db')
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO Pacjent (imie, nazwisko, pesel, data_urodzenia, telefon, email) VALUES (?, ?, ?, ?, ?, ?)",
                       (imie, nazwisko, pesel, data_urodzenia, telefon, email))
        conn.commit()
        print(f"Pacjent {imie} {nazwisko} ({pesel}) został dodany.")
        return True
    except sqlite3.IntegrityError:
        print(f"Błąd: Pacjent z peselem {pesel} już istnieje.")
        return False
    finally:
        conn.close()

# Przykłady użycia:
add_patient('Jan', 'Kowalski', '12345678901', '1990-01-01', '123456789', 'jan.kowalski@example.com')
add_patient('Anna', 'Nowak', '98765432109', '1985-05-15', '987654321', 'anna.nowak@example.com')
add_patient('Jan', 'Kowalski', '12345678901', '1990-01-01', '123456789', 'jan.kowalski@example.com') # Próba dodania duplikatu

Pacjent Jan Kowalski (12345678901) został dodany.
Pacjent Anna Nowak (98765432109) został dodany.
Błąd: Pacjent z peselem 12345678901 już istnieje.


False

### 3. Formularz i mechanizm logowania pacjenta

Oto formularz logowania HTML oraz funkcja Pythona do weryfikacji danych logowania.

```html
<!DOCTYPE html>
<html>
<head>
    <title>Logowanie Pacjenta</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; }
        .container { background-color: #fff; padding: 20px; border-radius: 8px; box-shadow: 0 0 10px rgba(0,0,0,0.1); max-width: 500px; margin: auto; }
        h2 { color: #333; }
        label { display: block; margin-bottom: 8px; font-weight: bold; }
        input[type="text"] { width: calc(100% - 22px); padding: 10px; margin-bottom: 15px; border: 1px solid #ddd; border-radius: 4px; }
        input[type="submit"] { background-color: #007bff; color: white; padding: 10px 15px; border: none; border-radius: 4px; cursor: pointer; font-size: 16px; }
        input[type="submit"]:hover { background-color: #0056b3; }
    </style>
</head>
<body>
    <div class="container">
        <h2>Logowanie Pacjenta</h2>
        <form action="/login_pesel" method="post">
            <label for="pesel">PESEL:</label>
            <input type="text" id="pesel" name="pesel" required>

            <input type="submit" value="Zaloguj (tylko PESEL)">
        </form>
        <p><i>Uwaga: Nowy schemat tabeli 'Pacjent' nie zawiera pola 'hasło'. Logowanie odbywa się tylko na podstawie PESELu. Aby dodać pełne logowanie z hasłem, należałoby dodać pole 'hasło' do tabeli 'Pacjent' lub stworzyć osobną tabelę użytkowników.</i></p>
    </div>
</body>
</html>
```

In [9]:
def verify_login_pesel(pesel):
    conn = sqlite3.connect('patients.db')
    cursor = conn.cursor()
    # W nowym schemacie tabeli 'Pacjent' nie ma pola 'password'.
    # Logowanie będzie polegać na sprawdzeniu istnienia PESELu.
    # Aby zaimplementować pełne logowanie z hasłem, należałoby dodać pole 'haslo' do tabeli Pacjent
    # i hashować je przed zapisem, tak jak to było w poprzedniej wersji.
    cursor.execute("SELECT pacjent_id, imie, nazwisko FROM Pacjent WHERE pesel = ?", (pesel,))
    result = cursor.fetchone()
    conn.close()

    if result:
        patient_id, imie, nazwisko = result
        print(f"Logowanie udane dla pacjenta {imie} {nazwisko} z PESEL: {pesel}")
        return patient_id
    print(f"Logowanie nieudane dla PESEL: {pesel} - brak pacjenta o podanym PESELu.")
    return None

# Przykłady użycia:
patient_id_jan = verify_login_pesel('12345678901')
patient_id_anna = verify_login_pesel('98765432109')
verify_login_pesel('00000000000') # Próba logowania nieistniejącego PESELu

Logowanie udane dla pacjenta Jan Kowalski z PESEL: 12345678901
Logowanie udane dla pacjenta Anna Nowak z PESEL: 98765432109
Logowanie nieudane dla PESEL: 00000000000 - brak pacjenta o podanym PESELu.


### 4. Prezentacja w HTML danych pacjenta (Karta pacjenta)

Poniżej znajduje się funkcja do pobierania danych pacjenta z bazy oraz szablon HTML do ich wyświetlania.

In [10]:
def get_patient_data(pacjent_id):
    conn = sqlite3.connect('patients.db')
    cursor = conn.cursor()
    cursor.execute("SELECT imie, nazwisko, pesel, data_urodzenia, telefon, email FROM Pacjent WHERE pacjent_id = ?", (pacjent_id,))
    patient_data = cursor.fetchone()
    conn.close()
    return patient_data

def generate_patient_card_html(patient_data):
    if not patient_data:
        return "<p>Nie znaleziono danych pacjenta.</p>"

    imie, nazwisko, pesel, data_urodzenia, telefon, email = patient_data
    html_output = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Karta Pacjenta</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; }}
        .card {{ background-color: #fff; padding: 20px; border-radius: 8px; box-shadow: 0 0 10px rgba(0,0,0,0.1); max-width: 600px; margin: auto; }}
        h2 {{ color: #333; border-bottom: 2px solid #007bff; padding-bottom: 10px; }}
        p {{ margin-bottom: 10px; font-size: 1.1em; }}
        p strong {{ color: #007bff; }}
    </style>
</head>
<body>
    <div class="card">
        <h2>Karta Pacjenta</h2>
        <p><strong>Imię:</strong> {imie}</p>
        <p><strong>Nazwisko:</strong> {nazwisko}</p>
        <p><strong>PESEL:</strong> {pesel}</p>
        <p><strong>Data Urodzenia:</strong> {data_urodzenia if data_urodzenia else 'Brak danych'}</p>
        <p><strong>Telefon:</strong> {telefon if telefon else 'Brak danych'}</p>
        <p><strong>Email:</strong> {email if email else 'Brak danych'}</p>
        <!-- Tutaj można dodać więcej danych pacjenta, np. historię wizyt itp. -->
    </div>
</body>
</html>
"""
    return html_output

# Przykład użycia (po udanym logowaniu):
if patient_id_jan:
    jan_data = get_patient_data(patient_id_jan)
    if jan_data:
        print("\n--- Karta Pacjenta (Jan Kowalski) ---")
        print(generate_patient_card_html(jan_data))
    else:
        print("Nie udało się pobrać danych dla Jana Kowalskiego.")

if patient_id_anna:
    anna_data = get_patient_data(patient_id_anna)
    if anna_data:
        print("\n--- Karta Pacjenta (Anna Nowak) ---")
        print(generate_patient_card_html(anna_data))
    else:
        print("Nie udało się pobrać danych dla Anny Nowak.")


--- Karta Pacjenta (Jan Kowalski) ---

<!DOCTYPE html>
<html>
<head>
    <title>Karta Pacjenta</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; }
        .card { background-color: #fff; padding: 20px; border-radius: 8px; box-shadow: 0 0 10px rgba(0,0,0,0.1); max-width: 600px; margin: auto; }
        h2 { color: #333; border-bottom: 2px solid #007bff; padding-bottom: 10px; }
        p { margin-bottom: 10px; font-size: 1.1em; }
        p strong { color: #007bff; }
    </style>
</head>
<body>
    <div class="card">
        <h2>Karta Pacjenta</h2>
        <p><strong>Imię:</strong> Jan</p>
        <p><strong>Nazwisko:</strong> Kowalski</p>
        <p><strong>PESEL:</strong> 12345678901</p>
        <p><strong>Data Urodzenia:</strong> 1990-01-01</p>
        <p><strong>Telefon:</strong> 123456789</p>
        <p><strong>Email:</strong> jan.kowalski@example.com</p>
        <!-- Tutaj można dodać więcej danych pacjenta, np. historię wizy